In [1]:
import os
import time
import tiktoken
from openai import OpenAI
from dotenv import load_dotenv
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


from datasets import load_dataset

In [2]:
ds = load_dataset("KushT/bbc_news_multiclass_train_val_test")

test = ds['test'].to_pandas()

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 334 entries, 0 to 333
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    334 non-null    object
 1   label   334 non-null    int64 
dtypes: int64(1), object(1)
memory usage: 5.3+ KB


In [3]:
test['label'] = test['label'].apply(lambda x: 'business' if x == 0 else 'entertainment' if x == 1 else 'politics' if x == 2 else 'sport' if x == 3 else 'tech')

labels = test['label'].unique()

test

,text,label
0,Dogged Federer claims Dubai crown World number...,sport
1,UK troops on Ivory Coast standby Downing Stree...,politics
2,Keanu Reeves given Hollywood star Actor Keanu ...,entertainment
3,Classy Henman makes winning start Tim Henman o...,sport
4,Mixed reaction to Man Utd offer Shares in Manc...,business
...,...,...
329,Middlesbrough 2-2 Charlton A late header by te...,sport
330,Budget Aston takes on Porsche British car make...,business
331,Hi-tech posters guide commuters Interactive po...,tech
332,Hotspot users gain free net calls People using...,tech


In [4]:
load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [5]:
def classify(text, labels):
    start_time = time.time()

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        store=True,
        messages = [
            {"role": "system", "content": "You are a classification assistant. Your objective is to read the provided text and classify it according to the task and labels described. You are capable of handling multiclass classification tasks based on user instructions."},
            {"role": "user", "content": f"Classify the following text based on the task: Category classification of news articles. Only respond with the label that best describe the text. The possible labels are: {', '.join(labels)}. Text: {text}"}
        ],
    )

    request_time = time.time() - start_time
    completion = response.choices[0].message.content.lower()
    completion_tokens = response.usage.completion_tokens
    prompt_tokens = response.usage.prompt_tokens
    total_tokens = response.usage.total_tokens

    return completion, request_time, completion_tokens, prompt_tokens, total_tokens

def post_process(text):
    if 'business' in text:
        return 'business'
    if 'entertainment' in text:
        return 'entertainment'
    if 'politics' in text:
        return 'politics'
    if 'sport' in text:
        return 'sport'
    if 'tech' in text:
        return 'tech'
    else:
        return 'error'

In [6]:
pred_df = test.copy() 

for index, row in pred_df.iterrows():
    try:
        text = row['text']
        completion, request_time, completion_tokens, prompt_tokens, total_tokens = classify(text, labels)
        pred_df.at[index, 'prediction'] = completion
        pred_df.at[index, 'request_time'] = request_time
        pred_df.at[index, 'completion_tokens'] = completion_tokens
        pred_df.at[index, 'prompt_tokens'] = prompt_tokens
        pred_df.at[index, 'total_tokens'] = total_tokens

    except Exception as e:
        # Save the current state of the DataFrame to a file before breaking out or retrying.
        pred_df.to_csv("results/partial_openai_ZS_multiclass2.csv", index=False)
        print(f"An error occurred at index {index}: {e}. Partial results saved.")
        # Optionally, you can break out of the loop or continue based on your needs.
        break

pred_df['prediction_post_processed'] = pred_df['prediction'].apply(post_process)
pred_df.to_csv("results/openai_ZS_multiclass2.csv", index=False)

pred_df

,text,label,prediction,request_time,completion_tokens,prompt_tokens,total_tokens,prediction_post_processed
0,Dogged Federer claims Dubai crown World number...,sport,sport,1.688727,2.0,513.0,515.0,sport
1,UK troops on Ivory Coast standby Downing Stree...,politics,politics,0.558286,3.0,352.0,355.0,politics
2,Keanu Reeves given Hollywood star Actor Keanu ...,entertainment,entertainment,0.587840,4.0,371.0,375.0,entertainment
3,Classy Henman makes winning start Tim Henman o...,sport,sport,0.449035,2.0,620.0,622.0,sport
4,Mixed reaction to Man Utd offer Shares in Manc...,business,business,0.505768,2.0,707.0,709.0,business
...,...,...,...,...,...,...,...,...
329,Middlesbrough 2-2 Charlton A late header by te...,sport,sport,0.644240,2.0,914.0,916.0,sport
330,Budget Aston takes on Porsche British car make...,business,business,0.465904,2.0,374.0,376.0,business
331,Hi-tech posters guide commuters Interactive po...,tech,tech,0.943535,2.0,486.0,488.0,tech
332,Hotspot users gain free net calls People using...,tech,tech,0.847988,2.0,402.0,404.0,tech


In [7]:
pred_df

,text,label,prediction,request_time,completion_tokens,prompt_tokens,total_tokens,prediction_post_processed
0,Dogged Federer claims Dubai crown World number...,sport,sport,1.688727,2.0,513.0,515.0,sport
1,UK troops on Ivory Coast standby Downing Stree...,politics,politics,0.558286,3.0,352.0,355.0,politics
2,Keanu Reeves given Hollywood star Actor Keanu ...,entertainment,entertainment,0.587840,4.0,371.0,375.0,entertainment
3,Classy Henman makes winning start Tim Henman o...,sport,sport,0.449035,2.0,620.0,622.0,sport
4,Mixed reaction to Man Utd offer Shares in Manc...,business,business,0.505768,2.0,707.0,709.0,business
...,...,...,...,...,...,...,...,...
329,Middlesbrough 2-2 Charlton A late header by te...,sport,sport,0.644240,2.0,914.0,916.0,sport
330,Budget Aston takes on Porsche British car make...,business,business,0.465904,2.0,374.0,376.0,business
331,Hi-tech posters guide commuters Interactive po...,tech,tech,0.943535,2.0,486.0,488.0,tech
332,Hotspot users gain free net calls People using...,tech,tech,0.847988,2.0,402.0,404.0,tech


In [8]:
y_pred = pred_df['prediction_post_processed']
y_true = pred_df['label']

#calculate the accuracy of the model
accuracy = accuracy_score(y_true, y_pred)
print('Accuracy: %f' % accuracy)
f1 = f1_score(y_true, y_pred, average='weighted')
print('F1 score: %f' % f1)
precision = precision_score(y_true, y_pred, average='weighted')
print('Precision: %f' % precision)
recall = recall_score(y_true, y_pred, average='weighted')
print('Recall: %f' % recall)

Accuracy: 0.931138
F1 score: 0.929036
Precision: 0.934674
Recall: 0.931138


In [9]:
# get average response time, vram usage and ram usage
request_time_avg = pred_df['request_time'].mean()
completion_tokens_avg = pred_df['completion_tokens'].mean()
prompt_tokens_avg = pred_df['prompt_tokens'].mean()
total_tokens_avg = pred_df['total_tokens'].mean()

print(f'Average response time: {request_time_avg}')
print(f'Average completion tokens: {completion_tokens_avg}')
print(f'Average prompt tokens: {prompt_tokens_avg}')
print(f'Average total tokens: {total_tokens_avg}')

Average response time: 0.6904944752504726
Average completion tokens: 2.5808383233532934
Average prompt tokens: 566.9730538922156
Average total tokens: 569.5538922155689


In [10]:
input_token_price = 0.15/1_000_000
output_token_price = 0.6/1_000_000

def count_tokens(text, model="gpt-4o-mini"):
    try:
        # Try to get the encoding for the given model
        encoding = tiktoken.encoding_for_model(model)
    except KeyError:
        # If the model isn't recognized, fall back to a default encoding
        encoding = tiktoken.get_encoding("cl100k_base")
    
    tokens = encoding.encode(text)
    return len(tokens)

# Calculate the cost of the requests
total_cost = 0
for index, row in pred_df.iterrows():
    completion_tokens = row['completion_tokens']
    prompt_tokens = row['prompt_tokens']
    cost  = completion_tokens * output_token_price + prompt_tokens * input_token_price
    total_cost += cost

print(f'Total cost: USD {total_cost}')

Total cost: USD 0.028922549999999984


In [ ]:
with open('results/openai_ZS_multiclass2.txt', 'w') as f:
    f.write(f'Accuracy: {accuracy}\n')
    f.write(f'F1 score: {f1}\n')
    f.write(f'Precision: {precision}\n')
    f.write(f'Recall: {recall}\n')
    f.write(f'Average response time: {request_time_avg}\n')
    f.write(f'Average completion tokens: {completion_tokens_avg}\n')
    f.write(f'Average prompt tokens: {prompt_tokens_avg}\n')
    f.write(f'Average total tokens: {total_tokens_avg}\n')
    f.write(f'Total cost: USD {total_cost}\n')